# Fast GECToR と W&I による英語誤り検出・訂正

書籍 3-4「演習：文法誤り検出・訂正」に対応します。

このノートブックでは、[Fast GECToR](https://github.com/cofe-ai/fast-gector) という文法誤り検出・訂正システムを、英語学習者の作文に誤りの訂正を付けたコーパスで訓練し、英語の誤り訂正をしてみます。

データには、[BEA-2019 共有タスク](https://www.cl.cam.ac.uk/research/nl/bea2019st/)で公開された Write & Improve（W&I）コーパスと LOCNESS コーパス（W&I+LOCNESS v2.1）を使います。W&I は Cambridge English Write & Improve という英作文の学習サイトに投稿された作文に、誤りの訂正を付けたコーパスです。非営利の研究・教育目的なら、申請なしでダウンロードして使えます（ダウンロードした時点で、データに含まれるライセンスに同意したことになります）。

GPU を使うので、[ランタイム] → [ランタイムのタイプを変更] で [ハードウェア アクセラレータ] を `GPU`（T4 など）にしておいてください。

In [1]:
import torch

if torch.cuda.is_available():
    print("使用可能な GPU:", torch.cuda.get_device_name(0))
else:
    print("警告: GPU が見つかりません（CPU で実行されます）。Colab の「ランタイム」→「ランタイムのタイプを変更」から GPU を選択してください。")

使用可能な GPU: NVIDIA L4


## インストール

Fast GECToR は GECToR の訓練を高速に行う派生版のシステムです。GitHub でソースコードが公開されているので、手元にダウンロードして使いましょう。

In [2]:
%cd /content
![ -d fast-gector ] || git clone https://github.com/cofe-ai/fast-gector
%cd /content/fast-gector

/content
Cloning into 'fast-gector'...
/content/fast-gector


Fast GECToR には、訓練の最初の数エポックだけエンコーダ（RoBERTa）を固定して出力層だけを訓練する機能（`--cold_step_count`）があります。ただし、固定を解除するときに DeepSpeed の設定ファイルから学習率を読み込む場所が間違っていて、エラーになってしまいます。そこで、ソースコードを1行だけ修正しておきます。

In [3]:
!sed -i 's/self.lr = config\["optimizer"\].get("lr")/self.lr = config["optimizer"]["params"]["lr"]/' src/trainer.py
with open("src/trainer.py", encoding="utf-8") as f:
    trainer_src = f.read()
assert 'self.lr = config["optimizer"]["params"]["lr"]' in trainer_src, (
    "src/trainer.py のパッチ適用に失敗しました（上流コードが変更された可能性があります）"
)
!grep -n "self.lr = " src/trainer.py

145:        self.lr = config["optimizer"]["params"]["lr"]


Fast GECToR は2023年以降更新されておらず、少し古いバージョンの PyTorch（2.0）や DeepSpeed（0.10）、Transformers（4.31）を必要とします。2026年10月時点の Google Colab の Python は 3.13 で、これらのライブラリは Python 3.13 では動かないので、[uv](https://docs.astral.sh/uv/) という高速なパッケージ管理ツールを使って Python 3.10 の仮想環境を `/content/gector-env` に作り、そこにインストールします。数分かかります。最後に `True` と表示されれば GPU が使えます。

なお、以前のバージョンのノートブックでは NVIDIA の apex もインストールしていましたが、Fast GECToR では使われていないので、インストールしないことにしました。

In [4]:
!pip install -q uv
!uv venv -q --clear -p 3.10 /content/gector-env
# DeepSpeed はインストール時に PyTorch を使うので、先に PyTorch を入れておく
!uv pip install -q -p /content/gector-env torch==2.0.1 numpy==1.25.1 "setuptools<70" wheel packaging ninja
!uv pip install -q -p /content/gector-env --no-build-isolation deepspeed==0.10.0
!uv pip install -q -p /content/gector-env -r requirements.txt
!/content/gector-env/bin/python -c "import sys, torch, deepspeed, transformers; print(sys.version, torch.__version__, deepspeed.__version__, transformers.__version__, torch.cuda.is_available())"

[2026-11-25 01:33:57,805] [INFO] [real_accelerator.py:133:get_accelerator] Setting ds_accelerator to cuda (auto detect)


3.10.22 (main, Oct  9 2026, 15:21:30) [Clang 22.1.3 ] 2.0.1+cu117 0.10.0 4.31.0 True


以下では `!deepspeed ...` のようにコマンドを実行するので、作った仮想環境のコマンドが使えるように PATH を設定しておきます。ランタイムを再起動した場合は、このセルをもう一度実行してください。

In [5]:
%cd /content/fast-gector
import os
if "/content/gector-env/bin" not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = "/content/gector-env/bin:" + os.environ["PATH"]
!which deepspeed

/content/fast-gector
/content/gector-env/bin/deepspeed


## データの準備

W&I+LOCNESS v2.1 をダウンロードして展開します。ライセンスは `wi+locness/licence.wi.txt` と `wi+locness/license.locness.txt` に書かれています。

In [6]:
!curl -fsSLO https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz
!tar xzf wi+locness_v2.1.bea19.tar.gz
!ls wi+locness/m2

ABCN.dev.gold.bea19.m2	 A.train.gold.bea19.m2	C.dev.gold.bea19.m2
ABC.train.gold.bea19.m2  B.dev.gold.bea19.m2	C.train.gold.bea19.m2
A.dev.gold.bea19.m2	 B.train.gold.bea19.m2	N.dev.gold.bea19.m2


誤りの訂正は M2 という形式で書かれています。`S` で始まる行が学習者の書いた文（単語分割済み）、`A` で始まる行がその文に対する訂正（何単語目から何単語目までを何に置き換えるか）です。

In [7]:
!head -n 12 wi+locness/m2/ABC.train.gold.bea19.m2

S My town is a medium size city with eighty thousand inhabitants .
A 5 6|||R:OTHER|||- sized|||REQUIRED|||-NONE-|||0

S It has a high density population because its small territory .
A 4 4|||M:PUNCT|||-|||REQUIRED|||-NONE-|||0
A 7 7|||M:PREP|||of|||REQUIRED|||-NONE-|||0

S Despite of it is an industrial city , there are many shops and department stores .
A 0 1|||R:PREP|||Although|||REQUIRED|||-NONE-|||0
A 1 2|||U:PREP||||||REQUIRED|||-NONE-|||0

S I recommend visiting the artificial lake in the certer of the city which is surrounded by a park .


Fast GECToR の訓練には「誤りを含む文」と「訂正後の文」のペアが必要なので、M2 の訂正を適用して訂正後の文を作ります。

W&I+LOCNESS の訓練データ（`ABC.train`、約3.4万文）を訓練データにします。テストデータの正解は公開されていないので、開発データ（`ABCN.dev`、約4,400文）をシャッフルして半分ずつに分け、開発データとテストデータにします。

In [8]:
import random

def m2_to_parallel(path):
    """M2 ファイルを（誤りを含む文, 訂正後の文）のペアのリストに変換する"""
    pairs = []
    for block in open(path, encoding="utf-8").read().strip().split("\n\n"):
        lines = block.split("\n")
        src = lines[0][2:].split()  # 先頭の "S " を除く
        trg = list(src)
        offset = 0  # これまでの訂正で増減した単語数
        for line in lines[1:]:
            fields = line[2:].split("|||")  # 先頭の "A " を除く
            start, end = map(int, fields[0].split())
            error_type, correction, annotator = fields[1], fields[2], fields[5]
            if error_type in ("noop", "UNK") or annotator != "0":
                continue
            cor_tokens = correction.split()
            trg[start + offset:end + offset] = cor_tokens
            offset += len(cor_tokens) - (end - start)
        pairs.append((" ".join(src), " ".join(trg)))
    return pairs

train = m2_to_parallel("wi+locness/m2/ABC.train.gold.bea19.m2")
dev_test = m2_to_parallel("wi+locness/m2/ABCN.dev.gold.bea19.m2")
random.seed(0)
random.shuffle(dev_test)
half = len(dev_test) // 2
splits = {"train": train, "dev": dev_test[:half], "test": dev_test[half:]}

for name, pairs in splits.items():
    with open(f"{name}.src", "w", encoding="utf-8") as fs, open(f"{name}.trg", "w", encoding="utf-8") as ft:
        for src, trg in pairs:
            fs.write(src + "\n")
            ft.write(trg + "\n")
    print(name, len(pairs), "文（うち誤りを含む文", sum(src != trg for src, trg in pairs), "）")

train 34308 文（うち誤りを含む文 22737 ）
dev 2192 文（うち誤りを含む文 1406 ）
test 2192 文（うち誤りを含む文 1413 ）


In [9]:
!paste train.src train.trg | head -n 5

My town is a medium size city with eighty thousand inhabitants .	My town is a medium - sized city with eighty thousand inhabitants .
It has a high density population because its small territory .	It has a high - density population because of its small territory .
Despite of it is an industrial city , there are many shops and department stores .	Although it is an industrial city , there are many shops and department stores .
I recommend visiting the artificial lake in the certer of the city which is surrounded by a park .	I recommend visiting the artificial lake in the center of the city which is surrounded by a park .
Pasteries are very common and most of them offer the special dessert from the city .	Pasteries are very common and most of them offer the special dessert of the city .


GECToR は入力と出力のペアから編集の系列を抽出し、タグとして用います。Fast GECToR にはこの編集系列の抽出のためのスクリプトが含まれているので、こちらを使いましょう（上で作った仮想環境の Python で実行します）。数秒で終わります。

In [10]:
# preprocess_data.py は出力ファイルに追記するので、セルを再実行したときのために先に消しておく
!rm -f train.edits dev.edits
!python utils/preprocess_data.py -s train.src -t train.trg -o train.edits
!python utils/preprocess_data.py -s dev.src -t dev.trg -o dev.edits
!wc -l train.edits dev.edits test.src test.trg

The size of raw dataset is 34304



Overall extracted 34304. Original TP 22733. Original TN 11571


The size of raw dataset is 2192



Overall extracted 2192. Original TP 1406. Original TN 786


   34304 train.edits
    2192 dev.edits
    2192 test.src
    2192 test.trg
   40880 total


## モデルの訓練

作成したデータを用いて Fast GECToR モデルの訓練を行います。Fast GECToR は DeepSpeed というライブラリを使って訓練します。DeepSpeed の設定ファイルは `configs/` の下にありますが、Google Colab の GPU（T4）向けに以下のように変更した設定ファイルを用意します。

- `train_batch_size` は256のままで、`gradient_accumulation_steps` を8にして、一度に GPU に載せる文の数を32文にします（GPU のメモリが足りなくならないようにするため）。
- `torch_adam` を `true` にして、PyTorch の Adam を使います（DeepSpeed の Adam を使うと、Google Colab の CUDA のバージョンが合わずにコンパイルに失敗するため）。
- 学習率（`lr`）を元の設定の1e-5から2e-4に上げます。元の設定は、人工的に作った誤りデータで事前に訓練したモデルをさらに訓練するためのものです。この演習のように事前訓練なしで W&I だけで訓練する場合、1e-5 だとほとんど大文字・小文字の訂正しか覚えません。2026年10月に学習率を5e-5から5e-4まで変えて試したところ、2e-4 が一番良く、5e-4 にすると訓練がうまくいきませんでした。

In [11]:
%%writefile configs/ds_config_colab.json
{
    "train_batch_size": 256,
    "gradient_accumulation_steps": 8,
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 2e-4,
            "eps": 1e-8,
            "weight_decay": 1e-2,
            "torch_adam": true
        }
    },
    "gradient_clipping": 1.0,
    "fp16": {
        "enabled": true,
        "auto_cast": false,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "consecutive_hysteresis": false,
        "min_loss_scale": 1
    },
    "zero_optimization": {
        "stage": 1,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8
    }
}

Writing configs/ds_config_colab.json


訓練には `train.py` を用います（`scripts/train.sh` の中身とほぼ同じですが、Google Colab 向けにいくつかオプションを変えています）。

- `--num_epochs` は訓練のエポック数です。ここでは10エポックにしています。2026年10月に試したときは、T4 で40分ほどかかりました（最初の2エポックはエンコーダを固定するので速く進みます）。
- `--cold_step_count` は、最初の何エポックの間エンコーダを固定して、新しく追加した出力層だけを `--cold_lr` の学習率で訓練するかを指定します。出力層はランダムな値から始まるので、最初からエンコーダも一緒に訓練するより、出力層をある程度訓練してからエンコーダも訓練するほうが良い結果になります。ここでは GECToR の訓練ガイドと同じく2エポックにしています。
- `--eval_interval` ステップごとに開発データで評価し、`--save_interval` ステップごとにモデルを `ckpts/` の下に保存します。両方を同じ値にしておくと、開発データで一番良かったモデルが必ず保存されます。
- `--tn_prob` は「訂正の必要がない文を訓練に使う確率」をコントロールします。これを変化させることで precision/recall のバランスを取ることができます。いくつかのステージに分けて訓練し、最初の訓練時には `--tn_prob 0` として、最後に適切な値に調整すると良いようです。また、`--skip_correct` もデフォルトは0ですが1にした方が良いようです。詳しくは [GECToR の訓練ガイド](https://github.com/grammarly/gector/blob/master/docs/training_parameters.md)をご覧ください。

訓練の様子は TensorBoard で見ることができます。

### 実行モードの切り替え（`QUICK_RUN`）

まずは短時間で動作確認したい場合は、下のセルで `QUICK_RUN = True` に変更してください。訓練データを 5,000 文・3 エポックに絞って実行するため、T4 でも 5〜10 分ほどで訓練が終わります。本格的な評価を行う場合は `QUICK_RUN = False`（W&I 全 3.4 万文・10 エポック、T4 で約 40 分）のまま実行してください。

In [12]:
import os

# True: 5,000 文・3 エポックの軽量モード（T4 で約 5〜10 分）
# False: W&I 全 3.4 万文・10 エポックの通常モード（T4 で約 40 分）
QUICK_RUN = False

if QUICK_RUN:
    with open("train.edits", encoding="utf-8") as fin, open("train_quick.edits", "w", encoding="utf-8") as fout:
        for i, line in enumerate(fin):
            if i >= 5000:
                break
            fout.write(line)
    os.environ["TRAIN_FILE"] = "train_quick.edits"
    os.environ["NUM_EPOCHS"] = "3"
    os.environ["COLD_STEP_COUNT"] = "1"
    # 軽量モードでも複数のチェックポイントを比較できる間隔にする。
    os.environ["EVAL_INTERVAL"] = "15"
else:
    os.environ["TRAIN_FILE"] = "train.edits"
    os.environ["NUM_EPOCHS"] = "10"
    os.environ["COLD_STEP_COUNT"] = "2"
    os.environ["EVAL_INTERVAL"] = "100"

print(f"QUICK_RUN={QUICK_RUN} | TRAIN_FILE={os.environ['TRAIN_FILE']} | NUM_EPOCHS={os.environ['NUM_EPOCHS']}")

QUICK_RUN=False | TRAIN_FILE=train.edits | NUM_EPOCHS=10


In [13]:
%load_ext tensorboard
%tensorboard --logdir logs

In [14]:
%%bash
set -o pipefail
timestamp=`date "+%Y%m%d_%H%M%S"`
save_dir="ckpts/ckpt_$timestamp"
mkdir -p $save_dir
deepspeed --master_port 49828 train.py \
    --deepspeed \
    --deepspeed_config configs/ds_config_colab.json \
    --num_epochs "${NUM_EPOCHS:-10}" \
    --max_num_tokens 128 \
    --valid_batch_size 64 \
    --cold_step_count "${COLD_STEP_COUNT:-2}" \
    --warmup 0.1 \
    --cold_lr 1e-3 \
    --skip_correct 0 \
    --skip_complex 0 \
    --sub_token_mode average \
    --special_tokens_fix 1 \
    --unk2keep 0 \
    --tp_prob 1 \
    --tn_prob 1 \
    --detect_vocab_path ./data/vocabulary/d_tags.txt \
    --correct_vocab_path ./data/vocabulary/labels.txt \
    --do_eval \
    --train_path "${TRAIN_FILE:-train.edits}" \
    --valid_path dev.edits \
    --save_dir $save_dir \
    --use_cache 0 \
    --log_interval 10 \
    --eval_interval "${EVAL_INTERVAL:-100}" \
    --save_interval "${EVAL_INTERVAL:-100}" \
    --pretrained_transformer_path roberta-base \
    --tensorboard_dir logs/tb/gector_$timestamp \
    2>&1 | tee $save_dir/train.log | grep --line-buffered -v "it/s"

[2026-11-25 01:34:22,345] [INFO] [real_accelerator.py:133:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-11-25 01:34:23,232] [WARNING] [runner.py:196:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-11-25 01:34:23,294] [INFO] [runner.py:555:main] cmd = /content/gector-env/bin/python -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbM119 --master_addr=127.0.0.1 --master_port=49828 --enable_each_rank_log=None train.py --deepspeed --deepspeed_config configs/ds_config_colab.json --num_epochs 10 --max_num_tokens 128 --valid_batch_size 64 --cold_step_count 2 --warmup 0.1 --cold_lr 1e-3 --skip_correct 0 --skip_complex 0 --sub_token_mode average --special_tokens_fix 1 --unk2keep 0 --tp_prob 1 --tn_prob 1 --detect_vocab_path ./data/vocabulary/d_tags.txt --correct_vocab_path ./data/vocabulary/labels.txt --do_eval --train_path train.edits --valid_path dev.edits --save_dir ckpts/ckpt_20261125_013418 --use_cache

## モデルの評価

訓練したモデルを評価しましょう。訓練されたモデルは `ckpts/ckpt_時刻/globalstep-XXX` のような名前で保存されています。開発データでの学習曲線（TensorBoard）を見るとどのモデルが適しているかがわかります。ここでは、訓練中に記録された評価結果から、開発データでの正解率が一番高かったモデルを選びます。

In [15]:
import glob, json, os

valid_dirs = [d for d in sorted(glob.glob("ckpts/ckpt_*")) if glob.glob(f"{d}/metrics_globalstep-*.json")]
assert valid_dirs, "metrics_globalstep-*.json が見つかりません"
save_dir = valid_dirs[-1]
metrics_files = sorted(glob.glob(f"{save_dir}/metrics_globalstep-*.json"), key=lambda p: int(p.split("-")[-1].split(".")[0]))
best = json.load(open(metrics_files[-1]))
ckpt_path = f"{save_dir}/globalstep-{best['best_global_step']}"
print(best)
print(ckpt_path, os.path.exists(ckpt_path))
assert os.path.exists(ckpt_path), f"チェックポイントが見つかりません: {ckpt_path}"
os.environ["CKPT_PATH"] = ckpt_path

{'current_global_step': 1299, 'current_train_loss': 0.03657971076594973, 'valid_loss': 1.1767578125, 'valid_accuracy': 0.9154409152277411, 'best_global_step': 799, 'best_valid_loss': 0.66552734375, 'best_valid_accuracy': 0.9182358653686729}
ckpts/ckpt_20261125_013418/globalstep-799 True


`predict.py` でテストデータの誤りを訂正します（`scripts/predict.sh` の中身とほぼ同じです）。

`--min_error_probability` は、単語ごとの誤り検出スコアと編集候補のスコアに適用する閾値です。閾値を上げると通常、適合率が上がり、再現率が下がります。`--additional_confidence` は「訂正しない」というタグの確率に足す値で、これも大きくすると訂正が控えめになります。2026年10月に開発データで試したところ、`--min_error_probability 0.3`、`--additional_confidence 0` が一番良かったので、ここではその値を使います。

In [16]:
%%bash
set -euo pipefail
deepspeed --include localhost:0 --master_port 42991 predict.py \
    --batch_size 64 \
    --iteration_count 5 \
    --min_seq_len 3 \
    --max_num_tokens 128 \
    --min_error_probability 0.3 \
    --additional_confidence 0.0 \
    --sub_token_mode average \
    --max_pieces_per_token 5 \
    --ckpt_path "$CKPT_PATH" \
    --deepspeed_config configs/ds_config_colab.json \
    --detect_vocab_path ./data/vocabulary/d_tags.txt \
    --correct_vocab_path ./data/vocabulary/labels.txt \
    --pretrained_transformer_path roberta-base \
    --input_path test.src \
    --out_path test.pred \
    --special_tokens_fix 1 \
    --detokenize 0 \
    --segmented 1 \
    2>&1 | tail -n 5

model path: ckpts/ckpt_20261125_013418/globalstep-799
start predicting ...
35it [00:08,  3.96it/s]
total cost: 8.838151693344116s
[2026-11-25 01:54:44,061] [INFO] [launch.py:347:main] Process 739776 exits successfully.


入力（`test.src`）とシステムの出力（`test.pred`）の差分を見てみましょう。`-` で始まる行が入力、`+` で始まる行がシステムが訂正した文です。

In [17]:
!diff -u test.src test.pred | head -n 200

--- test.src	2026-11-25 01:34:05.286560819 +0900
+++ test.pred	2026-11-25 01:54:41.498876555 +0900
@@ -1,46 +1,46 @@
 I also worked for many theater companies and I am so good and kind when I talk with other people , also if I do n't know them .
-But television have also negative effects : First of all , children who spend a great deal of time watching television get less exercise .
-That is for these reasons I think I have the skills for the job .
-I took every free minute to drive snowboard .
+But television has negative effects : First of all , children who spend a great deal of time watching television get less exercise .
+That is for these reasons , I think I have the skills for the job .
+I took every free minute to drive the snowboard .
 With everyone moving at such a fast pace , the gaps are easy to miss , the voices of the mass media easily replace those voices that struggle to be heard .
-This is when an egg is removed from a woman placed in a test tube and then sperm is plac

### 訂正結果の誤り分析テーブル

`diff` だけでは正解（`test.trg`）との違いが追いにくいため、原文（`src`）・システム予測（`pred`）・正解（`trg`）を並べた表を作成し、各文を「完全一致訂正」「部分・過剰訂正」「見逃し」「正解通り無訂正」に分類して確認します。

In [18]:
import pandas as pd

with open("test.src", encoding="utf-8") as fs, open("test.pred", encoding="utf-8") as fp, open("test.trg", encoding="utf-8") as ft:
    srcs = [l.rstrip("\n") for l in fs]
    preds = [l.rstrip("\n") for l in fp]
    trgs = [l.rstrip("\n") for l in ft]

def classify_edit(src, pred, trg):
    if src == trg and pred == trg:
        return "正解通り無訂正"
    if src != trg and pred == trg:
        return "完全一致訂正"
    if src != trg and pred == src:
        return "見逃し"
    return "部分・過剰訂正"

df_err = pd.DataFrame({
    "原文 (src)": srcs,
    "システム予測 (pred)": preds,
    "正解 (trg)": trgs,
    "分類": [classify_edit(s, p, t) for s, p, t in zip(srcs, preds, trgs)],
})
print(df_err["分類"].value_counts())

for cat in ("完全一致訂正", "部分・過剰訂正", "見逃し"):
    print(f"\n=== {cat}（先頭 5 件）===")
    display(df_err[df_err["分類"] == cat].head(5))

分類
部分・過剰訂正    764
正解通り無訂正    713
見逃し        562
完全一致訂正     153
Name: count, dtype: int64

=== 完全一致訂正（先頭 5 件）===


,原文 (src),システム予測 (pred),正解 (trg),分類
23,"At the beginning Anna did not say anything , s...","At the beginning , Anna did not say anything ,...","At the beginning , Anna did not say anything ,...",完全一致訂正
32,The marsh pointed on the first chapter is good...,The marsh pointed in the first chapter is a go...,The marsh pointed in the first chapter is a go...,完全一致訂正
62,"I am interested in to join your group , becaus...","I am interested in joining your group , becaus...","I am interested in joining your group , becaus...",完全一致訂正
66,"Boxing is a dangerous sport , there are relati...",Boxing is a dangerous sport . There are relati...,Boxing is a dangerous sport . There are relati...,完全一致訂正
87,One could travel by boat but this was inconven...,One could travel by boat but this was inconven...,One could travel by boat but this was inconven...,完全一致訂正



=== 部分・過剰訂正（先頭 5 件）===


,原文 (src),システム予測 (pred),正解 (trg),分類
1,But television have also negative effects : Fi...,But television has negative effects : First of...,But television also has negative effects : Fir...,部分・過剰訂正
2,That is for these reasons I think I have the s...,"That is for these reasons , I think I have the...",It is for these reasons I think I have the ski...,部分・過剰訂正
3,I took every free minute to drive snowboard .,I took every free minute to drive the snowboard .,I took every free minute to go snowboarding .,部分・過剰訂正
5,This is when an egg is removed from a woman pl...,This is when an egg is removed from a woman pl...,"This is when an egg is removed from a woman , ...",部分・過剰訂正
6,"I can speak English and I am a lovely , energe...","I can speak English and I am a lovely , energe...","I can speak English and I am a lovely , energe...",部分・過剰訂正



=== 見逃し（先頭 5 件）===


,原文 (src),システム予測 (pred),正解 (trg),分類
0,I also worked for many theater companies and I...,I also worked for many theater companies and I...,I also worked for many theater companies and I...,見逃し
7,The cause for television violence is to add te...,The cause for television violence is to add te...,The reason for television violence is to add t...,見逃し
8,"Many people argue nowadays that , with the adv...","Many people argue nowadays that , with the adv...","Many people argue nowadays that , with the adv...",見逃し
13,People are now self - centred and very narrow ...,People are now self - centred and very narrow ...,People now are self - centred and very narrow ...,見逃し
14,I met many people in the park .,I met many people in the park .,I meet many people in the park .,見逃し


### ERRANT による評価

文法誤り訂正では、[ERRANT](https://github.com/chrisjbryant/errant) というツールで、システムの訂正と正解（参照訳）の訂正を比べて precision、recall、F0.5 を計算するのが一般的です。ERRANT をインストールして評価してみましょう。

下の実行結果（検証の GPU）では precision が0.53、recall が0.26、F0.5 が0.44でした（T4 で試したときは precision が0.51、recall が0.24、F0.5 が0.41程度で、実行のたびに結果は少し変わります）。最初のバージョンのノートブック（学習率5e-5、5エポック、`--cold_step_count 0`）では F0.5 が0.26～0.28程度だったので、学習率・エポック数・`--cold_step_count`・`--min_error_probability` を調整することで大きく改善しています。さらに、cLang-8 のような大きなデータで先に訓練してから W&I で訓練すると、もっと良くなります。

In [19]:
# ERRANT は Google Colab の Python にインストールする（PATH の先頭にある仮想環境の Python ではなく）
import sys
!{sys.executable} -m pip install -q errant
!{sys.executable} -m spacy download en_core_web_sm
!errant_parallel -orig test.src -cor test.pred -out hyp.m2
!errant_parallel -orig test.src -cor test.trg -out ref.m2
!errant_compare -hyp hyp.m2 -ref ref.m2

Loading resources...


Processing parallel files...


Loading resources...


Processing parallel files...



=========== Span-Based Correction ============
TP	FP	FN	Prec	Rec	F0.5
991	890	2784	0.5268	0.2625	0.4385



### 閾値（`--min_error_probability`）による Precision / Recall トレードオフ

文法誤り訂正では、正しい文を誤って書き換えてしまう（偽陽性）よりも見逃し（偽陰性）のほうが害が小さいため、`ch1_evaluation.ipynb` でも扱ったように 再現率より適合率を重視する $F_{0.5}$ が評価尺度として使われます。

軽量モードでは開発データの先頭 500 文、通常モードでは開発データ全体に対して `--min_error_probability` を `0.1, 0.3, 0.5, 0.7` と変えて推論し、ERRANT で Precision・Recall・$F_{0.5}$ がどのように変化するかを確認しましょう。

従来のチェックポイント（開発データで正解率が最高）に加え、同じ訓練で保存した他のチェックポイントから中ほどと最後のものも候補にします。すべての候補を同じ開発データの文で評価し、ERRANT の $F_{0.5}$ が最も高いチェックポイントと閾値の組を選びます。同点の場合は従来のチェックポイントを優先し、その次に従来の閾値 `0.3` に近いものを選びます。評価に使う文数は `N_DEV` で変更できます。

In [20]:
from itertools import islice
from pathlib import Path
import re
import subprocess
from tempfile import TemporaryDirectory

import pandas as pd


def run_prediction(checkpoint, threshold, source_path, pred_path):
    subprocess.run([
        "deepspeed", "--include", "localhost:0", "--master_port", "42992", "predict.py",
        "--batch_size", "64",
        "--iteration_count", "5",
        "--min_seq_len", "3",
        "--max_num_tokens", "128",
        "--min_error_probability", str(threshold),
        "--additional_confidence", "0.0",
        "--sub_token_mode", "average",
        "--max_pieces_per_token", "5",
        "--ckpt_path", str(checkpoint),
        "--deepspeed_config", "configs/ds_config_colab.json",
        "--detect_vocab_path", "./data/vocabulary/d_tags.txt",
        "--correct_vocab_path", "./data/vocabulary/labels.txt",
        "--pretrained_transformer_path", "roberta-base",
        "--input_path", str(source_path),
        "--out_path", str(pred_path),
        "--special_tokens_fix", "1",
        "--detokenize", "0",
        "--segmented", "1",
    ], check=True, stdout=subprocess.DEVNULL)


def annotate_edits(source_path, corrected_path, m2_path):
    subprocess.run([
        "errant_parallel", "-orig", str(source_path), "-cor", str(corrected_path),
        "-out", str(m2_path),
    ], check=True)


def errant_scores(hyp_path, ref_path):
    result = subprocess.run(
        ["errant_compare", "-hyp", str(hyp_path), "-ref", str(ref_path)],
        check=True, capture_output=True, text=True,
    )
    match = re.search(
        r"^\s*(\d+)\s+(\d+)\s+(\d+)\s+[0-9.]+\s+[0-9.]+\s+[0-9.]+\s*$",
        result.stdout, re.MULTILINE,
    )
    if match is None:
        raise ValueError(f"errant_compare の出力を解析できませんでした:\n{result.stdout}")
    tp, fp, fn = map(int, match.groups())
    # ERRANT の表示で丸められる前の F0.5 を、編集の件数から求める。
    return {
        "TP": tp, "FP": fp, "FN": fn,
        "Precision": tp / (tp + fp) if tp + fp else 0.0,
        "Recall": tp / (tp + fn) if tp + fn else 0.0,
        "F0.5": 5 * tp / (5 * tp + 4 * fp + fn) if tp + fp + fn else 0.0,
    }


def evaluate_prediction(checkpoint, threshold, source_path, ref_path, output_dir):
    pred_path = output_dir / "pred.txt"
    hyp_path = output_dir / "hyp.m2"
    run_prediction(checkpoint, threshold, source_path, pred_path)
    annotate_edits(source_path, pred_path, hyp_path)
    return errant_scores(hyp_path, ref_path)


baseline_checkpoint = Path(ckpt_path)
baseline_threshold = 0.3
other_checkpoints = sorted(
    (
        path for path in baseline_checkpoint.parent.glob("globalstep-*")
        if path.is_dir() and path.name.removeprefix("globalstep-").isdigit()
        and path != baseline_checkpoint
    ),
    key=lambda path: int(path.name.removeprefix("globalstep-")),
)
candidates = [baseline_checkpoint]
if other_checkpoints:
    middle = other_checkpoints[(len(other_checkpoints) - 1) // 2]
    candidates.extend(dict.fromkeys((middle, other_checkpoints[-1])))

# 評価する文と参照の M2 ファイルは、すべての候補で共通。
N_DEV = 500 if QUICK_RUN else None
dev_results = []
with TemporaryDirectory(prefix="gector-dev-") as directory:
    output_dir = Path(directory)
    dev_src, dev_trg = output_dir / "dev.src", output_dir / "dev.trg"
    for target in (dev_src, dev_trg):
        with open(target.name, encoding="utf-8") as lines:
            target.write_text("".join(islice(lines, N_DEV)), encoding="utf-8")
    dev_ref = output_dir / "ref.m2"
    annotate_edits(dev_src, dev_trg, dev_ref)
    for checkpoint in candidates:
        for threshold in (0.1, baseline_threshold, 0.5, 0.7):
            scores = evaluate_prediction(checkpoint, threshold, dev_src, dev_ref, output_dir)
            dev_results.append({
                "checkpoint": checkpoint.name,
                "min_error_probability": threshold,
                **scores,
            })

selected = max(dev_results, key=lambda row: (
    row["F0.5"],
    row["checkpoint"] == baseline_checkpoint.name,
    -abs(row["min_error_probability"] - baseline_threshold),
))
display(pd.DataFrame(dev_results).round(4))
print("開発データで選んだ設定:", selected["checkpoint"], selected["min_error_probability"])

,checkpoint,min_error_probability,TP,FP,FN,Precision,Recall,F0.5
0,globalstep-799,0.1,1005,1032,2803,0.4934,0.2639,0.4203
1,globalstep-799,0.3,994,931,2814,0.5164,0.2610,0.4319
2,globalstep-799,0.5,916,700,2892,0.5668,0.2405,0.4459
3,globalstep-799,0.7,732,450,3076,0.6193,0.1922,0.4288
4,globalstep-599,0.1,1118,1295,2690,0.4633,0.2936,0.4153
5,globalstep-599,0.3,1100,1148,2708,0.4893,0.2889,0.4297
6,globalstep-599,0.5,997,841,2811,0.5424,0.2618,0.4467
7,globalstep-599,0.7,735,472,3073,0.6089,0.1930,0.4255
8,globalstep-1299,0.1,1276,1596,2532,0.4443,0.3351,0.4171
9,globalstep-1299,0.3,1258,1393,2550,0.4745,0.3304,0.4364


開発データで選んだ設定: globalstep-1299 0.7


### 同じテストデータで従来設定と比較

上で選んだ組を固定し、従来設定（開発データの正解率が最高のチェックポイント、閾値 `0.3`）と、同じ `test.src` と参照 `ref.m2` で比較します。開発データでの改善がテストデータでも得られるか、適合率・再現率・$F_{0.5}$ を見比べてみましょう。

In [21]:
baseline_test = errant_scores("hyp.m2", "ref.m2")
selected_checkpoint = baseline_checkpoint.parent / selected["checkpoint"]
selected_threshold = selected["min_error_probability"]

if (selected_checkpoint, selected_threshold) == (baseline_checkpoint, baseline_threshold):
    selected_test = baseline_test
else:
    with TemporaryDirectory(prefix="gector-test-") as directory:
        selected_test = evaluate_prediction(
            selected_checkpoint, selected_threshold, Path("test.src"), Path("ref.m2"),
            Path(directory),
        )

pd.DataFrame([
    {
        "設定": "従来（開発正解率・閾値 0.3）",
        "checkpoint": baseline_checkpoint.name,
        "min_error_probability": baseline_threshold,
        **baseline_test,
    },
    {
        "設定": "開発 ERRANT F0.5 で選択",
        "checkpoint": selected_checkpoint.name,
        "min_error_probability": selected_threshold,
        **selected_test,
    },
]).round(4)

,設定,checkpoint,min_error_probability,TP,FP,FN,Precision,Recall,F0.5
0,従来（開発正解率・閾値 0.3）,globalstep-799,0.3,991,890,2784,0.5268,0.2625,0.4385
1,開発 ERRANT F0.5 で選択,globalstep-1299,0.7,1031,783,2744,0.5684,0.2731,0.4673


### 誤りタイプ別の評価と編集タグで表せる訂正（練習問題 3-3）

練習問題 3-3 の回答例では、GECToR のように各単語に編集タグを付ける方法は、訓練データに一定回数以上出現した訂正に出力を絞るので適合率は高くなるが、「訓練データに出現しない未知の誤りや、大幅な語順の入れ替え・書き換えを伴う誤りを訂正しにくいため、再現率に課題がある」と述べました。上の評価でも、再現率は適合率の半分ほどしかありません。ここでは、どんな誤りを見逃しているのかを、テストデータで確かめます。

1. ERRANT の `-cat` オプションで、誤りタイプ別の適合率・再現率・F0.5 を出し、再現率の低いタイプを調べる。
2. 正解の編集のうち、GECToR の編集タグの語彙（`data/vocabulary/labels.txt` の5,000種類）で、1回のタグ付けで表せるものが何割あるかを数える。

ERRANT の誤りタイプは、操作（`M` 欠落 missing、`R` 置換 replacement、`U` 不要 unnecessary）と、品詞などで分けた種類（`DET` 冠詞・限定詞、`PREP` 前置詞、`VERB:TENSE` 時制、`SPELL` つづり、`WO` 語順 word order、`OTHER` その他など）を組み合わせたものです。`errant_compare` に `-cat 1` を付けると操作ごと、`-cat 2` で種類ごと、`-cat 3` で両方を組み合わせたタイプ（`R:VERB:TENSE` など）ごとの集計になります。ERRANT で誤りタイプ付きの編集を取り出す方法は、クックブックの [ERRANT で英語の訂正を誤りタイプ付きの編集に変換する](https://github.com/mamorlis/nlpcookbook/blob/main/notebooks/29_gec/errant.ipynb) や [文法誤り訂正を編集単位の F0.5 で評価する](https://github.com/mamorlis/nlpcookbook/blob/main/notebooks/29_gec/gec_f05.ipynb) も参照してください。

まず、開発データで選んだ設定（上の表の2行目）の訂正結果を `test_selected.pred` に保存し、操作ごと（`-cat 1`）に集計します。

In [22]:
# 開発データで選んだ設定の訂正結果を、ファイルに残して使う
if (selected_checkpoint, selected_threshold) == (baseline_checkpoint, baseline_threshold):
    hyp_selected = "hyp.m2"
else:
    run_prediction(selected_checkpoint, selected_threshold, "test.src", "test_selected.pred")
    annotate_edits("test.src", "test_selected.pred", "hyp_selected.m2")
    hyp_selected = "hyp_selected.m2"
print(selected_checkpoint.name, selected_threshold, errant_scores(hyp_selected, "ref.m2"))
!errant_compare -hyp {hyp_selected} -ref ref.m2 -cat 1

globalstep-1299 0.7 {'TP': 1031, 'FP': 783, 'FN': 2744, 'Precision': 0.5683572216097024, 'Recall': 0.27311258278145695, 'F0.5': 0.4673193726770012}



===================== Span-Based Correction ======================
Category       TP       FP       FN       P        R        F0.5
M              398      258      645      0.6067   0.3816   0.5427
R              510      354      1836     0.5903   0.2174   0.4395
U              123      171      263      0.4184   0.3187   0.3937

=========== Span-Based Correction ============
TP	FP	FN	Prec	Rec	F0.5
1031	783	2744	0.5684	0.2731	0.4673



`M`（欠落）、`R`（置換）、`U`（不要）で比べると、どの操作の再現率が低いかが分かります。

次に `-cat 3` で、誤りタイプごとに集計します。タイプは50種類ほどあり、正解の編集が少ないタイプの値はあてにならないので、テストデータの正解の編集が30個以上あるタイプだけを、再現率の低い順に並べます。

In [23]:
def errant_scores_by_type(hyp_path, ref_path, cat):
    """errant_compare -cat の出力を、誤りタイプごとの DataFrame にする"""
    result = subprocess.run(
        ["errant_compare", "-hyp", str(hyp_path), "-ref", str(ref_path), "-cat", str(cat)],
        check=True, capture_output=True, text=True,
    )
    rows = []
    for line in result.stdout.splitlines():
        fields = line.split()
        # 「タイプ TP FP FN P R F0.5」の行だけを取り出す
        if len(fields) == 7 and all(f.isdigit() for f in fields[1:4]):
            tp, fp, fn = map(int, fields[1:4])
            rows.append({
                "タイプ": fields[0], "正解の編集数": tp + fn, "TP": tp, "FP": fp, "FN": fn,
                "Precision": tp / (tp + fp) if tp + fp else 0.0,
                "Recall": tp / (tp + fn) if tp + fn else 0.0,
                "F0.5": 5 * tp / (5 * tp + 4 * fp + fn) if tp + fp + fn else 0.0,
            })
    return pd.DataFrame(rows)


by_type = errant_scores_by_type(hyp_selected, "ref.m2", cat=3)
print("誤りタイプの数:", len(by_type))
frequent = by_type[by_type["正解の編集数"] >= 30].sort_values("Recall")
display(frequent.round(3).reset_index(drop=True))

誤りタイプの数: 54


,タイプ,正解の編集数,TP,FP,FN,Precision,Recall,F0.5
0,M:OTHER,73,2,3,71,0.400,0.027,0.108
1,R:SPELL,233,11,3,222,0.786,0.047,0.190
2,R:VERB,189,9,7,180,0.562,0.048,0.178
3,U:OTHER,40,2,13,38,0.133,0.050,0.100
4,R:WO,37,2,3,35,0.400,0.054,0.175
5,R:NOUN,137,8,5,129,0.615,0.058,0.212
6,R:ADJ,45,4,3,41,0.571,0.089,0.274
7,R:OTHER,296,27,34,269,0.443,0.091,0.250
8,R:MORPH,106,11,11,95,0.500,0.104,0.284
9,M:VERB,32,4,2,28,0.667,0.125,0.357


#### 1回のタグ付けで表せる訂正はどれくらいあるか

GECToR は、入力の各単語に、次のような編集タグを1つずつ付けて訂正します（`$START` という仮の単語を文頭に置き、文頭への挿入はそのタグで表します）。

- `$KEEP`（そのまま）、`$DELETE`（削除）
- `$APPEND_w`（後ろに単語 w を挿入）、`$REPLACE_w`（単語 w に置換）
- `$TRANSFORM_...`（大文字・小文字の変換、動詞の活用形の変換、単数形・複数形の変換、ハイフンでの分割）、`$MERGE_...`（後ろの単語との結合）

`APPEND` と `REPLACE` で使える単語は、訓練データでよく出てきた単語に限られます。タグの語彙は全部で5,000種類なので、それ以外の単語を挿入したり、それ以外の単語に置き換えたりすることはできません。また、モデルは1回の推論で1つの単語に1つのタグしか付けられません（Fast GECToR の訓練でも、1つの単語に複数のタグが必要な場合は、最初のタグだけを使います）。予測時は訂正した文をもう一度入力にして最大5回まで繰り返す（`--iteration_count 5`）ので、複数のタグが必要な訂正も直せることはありますが、訓練で学ぶのは1回目のタグだけです。

そこで、テストデータの正解（`ref.m2`）の編集を1つずつ原文に適用し、Fast GECToR が訓練データを作るのに使った `utils/preprocess_data.py` の `align_sequences` で編集タグに変換して、次の3つに分けます。

- **1回で表せる**：どの単語にもタグが1つだけで、すべてのタグが語彙にある
- **1つの単語に複数のタグが必要**：複数の単語の挿入（`$APPEND` が2つ以上）、2単語の入れ替えなど
- **語彙にないタグが必要**：訓練データにあまり出てこない単語への置換や、その単語の挿入

`align_sequences` は上で作った仮想環境のライブラリ（`Levenshtein` など）を使うので、スクリプトにして仮想環境の Python で実行します。

In [24]:
%%writefile tag_coverage.py
"""正解の M2 の編集を1つずつ GECToR の編集タグに変換し、1回のタグ付けで表せるかを調べる"""
import csv
import sys

sys.path.insert(0, "utils")
from helpers import SEQ_DELIMETERS
from preprocess_data import align_sequences

m2_path, vocab_path, out_path = sys.argv[1:]
vocab = set(open(vocab_path, encoding="utf-8").read().split())

rows = []
blocks = open(m2_path, encoding="utf-8").read().strip().split("\n\n")
for sent_id, block in enumerate(blocks):
    lines = block.split("\n")
    src = lines[0][2:].split()
    for line in lines[1:]:
        span, etype, cor = line[2:].split("|||")[:3]
        if etype == "noop":
            continue
        start, end = map(int, span.split())
        # この編集だけを原文に適用した文を作る
        trg = src[:start] + cor.split() + src[end:]
        tagged = align_sequences(" ".join(src), " ".join(trg))
        # 単語ごとのタグ（$KEEP 以外）を取り出す
        tags = []
        for i, token in enumerate(tagged.split(SEQ_DELIMETERS["tokens"])):
            word, labels = token.split(SEQ_DELIMETERS["labels"])
            labels = labels.split(SEQ_DELIMETERS["operations"])
            if labels != ["$KEEP"]:
                tags.append((word, labels))
        if any(len(labels) > 1 for _, labels in tags):
            status = "1つの単語に複数のタグが必要"
        elif any(labels[0] not in vocab for _, labels in tags):
            status = "語彙にないタグが必要"
        else:
            status = "1回で表せる"
        rows.append({
            "sent_id": sent_id, "start": start, "end": end, "type": etype,
            "orig": " ".join(src[start:end]), "cor": cor, "status": status,
            "tags": " ".join(f"{w}→{'+'.join(ls)}" for w, ls in tags),
        })

with open(out_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]), delimiter="\t")
    writer.writeheader()
    writer.writerows(rows)
print(len(rows), "個の編集を変換しました")

Writing tag_coverage.py


In [25]:
!python tag_coverage.py ref.m2 data/vocabulary/labels.txt tag_coverage.tsv

coverage = pd.read_csv("tag_coverage.tsv", sep="\t", keep_default_na=False)


def read_m2_edits(path):
    """M2 ファイルから {(文番号, 開始, 終了, 訂正)} の集合を作る"""
    edits = set()
    blocks = open(path, encoding="utf-8").read().strip().split("\n\n")
    for sent_id, block in enumerate(blocks):
        for line in block.split("\n")[1:]:
            span, etype, cor = line[2:].split("|||")[:3]
            if etype != "noop":
                start, end = map(int, span.split())
                edits.add((sent_id, start, end, cor))
    return edits


# ERRANT と同じく、位置と訂正が一致すればシステムが直せた（TP）とみなす
hyp_edits = read_m2_edits(hyp_selected)
coverage["直せた"] = [
    (r.sent_id, r.start, r.end, r.cor) in hyp_edits for r in coverage.itertuples()
]
assert coverage["直せた"].sum() == errant_scores(hyp_selected, "ref.m2")["TP"]

summary = coverage.groupby("status").agg(編集数=("直せた", "size"), 直せた数=("直せた", "sum"))
summary["割合"] = summary["編集数"] / summary["編集数"].sum()
summary["再現率"] = summary["直せた数"] / summary["編集数"]
display(summary.sort_values("編集数", ascending=False).round(3))

3775 個の編集を変換しました


,編集数,直せた数,割合,再現率
status,,,,
1回で表せる,3325,1030,0.881,0.310
語彙にないタグが必要,246,0,0.065,0.000
1つの単語に複数のタグが必要,204,1,0.054,0.005


誤りタイプごとに、1回のタグ付けで表せる編集の割合と再現率を並べてみます（正解の編集が30個以上あるタイプだけ、表せる割合の低い順）。

In [26]:
by_type_cov = coverage.groupby("type").agg(
    正解の編集数=("直せた", "size"),
    表せる割合=("status", lambda s: (s == "1回で表せる").mean()),
    複数タグ=("status", lambda s: (s == "1つの単語に複数のタグが必要").mean()),
    語彙外=("status", lambda s: (s == "語彙にないタグが必要").mean()),
    再現率=("直せた", "mean"),
)
by_type_cov = by_type_cov[by_type_cov["正解の編集数"] >= 30].sort_values("表せる割合")
display(by_type_cov.round(3))
print("表せる割合と再現率の相関係数:", round(by_type_cov["表せる割合"].corr(by_type_cov["再現率"]), 3))

,正解の編集数,表せる割合,複数タグ,語彙外,再現率
type,,,,,
M:OTHER,73,0.027,0.973,0.000,0.027
R:SPELL,233,0.614,0.000,0.386,0.047
R:OTHER,296,0.642,0.247,0.111,0.091
R:NOUN,137,0.701,0.029,0.270,0.058
R:MORPH,106,0.745,0.000,0.255,0.104
R:WO,37,0.757,0.108,0.135,0.054
R:VERB,189,0.810,0.063,0.127,0.048
M:VERB,32,0.812,0.125,0.062,0.125
R:ORTH,162,0.815,0.160,0.025,0.426


表せる割合と再現率の相関係数: 0.552


1回のタグ付けで表せない編集の例を見てみましょう。`tags` 列は、その編集を表すのに必要なタグです（`→` の左が単語、`+` は1つの単語に複数のタグが必要なことを表します）。

In [27]:
pd.set_option("display.max_colwidth", 80)
examples = pd.concat([
    coverage[coverage["type"] == "R:WO"].head(4),
    coverage[(coverage["status"] == "1つの単語に複数のタグが必要") & (coverage["type"] != "R:WO")].head(4),
    coverage[coverage["status"] == "語彙にないタグが必要"].head(4),
])
display(examples[["type", "orig", "cor", "status", "tags", "直せた"]].reset_index(drop=True))

,type,orig,cor,status,tags,直せた
0,R:WO,are now,now are,1回で表せる,People→$APPEND_now now→$DELETE,False
1,R:WO,Either you can,You can either,1回で表せる,Either→$DELETE you→$TRANSFORM_CASE_CAPITAL can→$APPEND_either,False
2,R:WO,", also","Also ,",1回で表せる,anatomy→$APPEND_Also also→$DELETE,False
3,R:WO,is our life,our life is,1回で表せる,is→$DELETE life→$APPEND_is,False
4,R:OTHER,besides,". In addition ,",1つの単語に複数のタグが必要,"besides→$REPLACE_.+$APPEND_In+$APPEND_addition+$APPEND_,",False
5,R:NOUN:POSS,mothers,mother 's,1つの単語に複数のタグが必要,mothers→$TRANSFORM_VERB_VBZ_VB+$APPEND_'s,False
6,R:OTHER,elevation,the rise,1つの単語に複数のタグが必要,elevation→$REPLACE_the+$APPEND_rise,False
7,R:OTHER,could,was able to,1つの単語に複数のタグが必要,I→$APPEND_was could→$REPLACE_able+$APPEND_to,False
8,R:NOUN,paper,reference,語彙にないタグが必要,paper→$REPLACE_reference,False
9,R:SPELL,adotpion,adoption,語彙にないタグが必要,adotpion→$REPLACE_adoption,False


#### 結果のまとめ

上の実行結果から、練習問題 3-3 の回答例を確かめましょう（数値は実行のたびに少し変わります）。

- **置換の再現率が低い**：操作ごとに見ると、再現率は欠落（`M`）が0.38、不要（`U`）が0.32なのに対し、置換（`R`）は0.22です。
- **再現率の低いタイプ**：`M:OTHER`（0.03）、`R:SPELL`（0.05）、`R:VERB`（0.05）、`U:OTHER`（0.05）、`R:WO`（語順、0.05）、`R:NOUN`（0.06）は、ほとんど直せていません。一方、主語と動詞の一致（`R:VERB:SVA`、0.61）、句読点の欠落（`M:PUNCT`、0.53）、動詞の形（`R:VERB:FORM`、0.45）、冠詞の欠落（`M:DET`、0.41）はよく直せています。時制（`R:VERB:TENSE`）は0.28で、その中間です。よく直せるタイプは、少数のタグ（`$APPEND_,`、`$APPEND_the`、`$TRANSFORM_VERB_...` など）で表せて、訓練データにも多く出てくる誤りです。
- **1回のタグ付けで表せない編集は、ほぼ直せない**：テストデータの正解の編集3,775個のうち、1回のタグ付けで表せるのは3,325個（88.1%）でした。残りの450個（11.9%）は、1つの単語に複数のタグが必要なもの（204個）と、語彙にないタグが必要なもの（246個）で、システムが直せたのは1個だけです（表せる編集の再現率は0.31）。複数語の挿入（`M:OTHER`）は97%が1回のタグ付けでは表せず、つづりの誤り（`R:SPELL`）は39%、名詞の置換（`R:NOUN`）は27%が、語彙にない単語への置換です。誤りタイプごとの「表せる割合」と再現率の相関係数は0.55でした。
- **語順の誤りは、表せても直しにくい**：2単語の入れ替え（`are now` → `now are`）は、`$DELETE` と `$APPEND_now` のように、離れた単語に付けた2つのタグの組み合わせで表せるので、`R:WO` の76%は「1回で表せる」に入ります。しかし、2つのタグをそろって正しく予測しないと訂正にならないので、再現率は0.05しかありません（`$MERGE_SWAP` という入れ替えのタグもありますが、Fast GECToR のタグの語彙には入っていません）。

このように、タグの語彙と1単語1タグという制約のために、訓練データにない単語への訂正や、複数語の書き換え、語順の入れ替えは直しにくく、再現率が下がることが確かめられました。ただし、見逃した編集（FN）2,744個のうち、約84%は1回のタグ付けで表せる編集でした。つづりや内容語（名詞・動詞）の置換、不要な語（`U:OTHER`）の削除は、タグでは表せても、どの単語をどう直すかを W&I の3.4万文だけから学ぶのは難しいようです。GECToR の論文では、人工的に作った大量の誤りデータで事前に訓練してから W&I などで訓練することで、性能を大きく上げています。

人工的な誤りデータの作り方はクックブックの [規則で誤りを注入して GEC の擬似学習データを作る](https://github.com/mamorlis/nlpcookbook/blob/main/notebooks/29_gec/synthetic_errors.ipynb)、学習済みの系列変換モデルによる英語の誤り訂正は [学習済みモデルで英語の文法誤りを訂正する](https://github.com/mamorlis/nlpcookbook/blob/main/notebooks/29_gec/gec_model_en.ipynb) も参照してください。